# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/si-ux/FlyrankAI-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

> **Lane:** Ranking Signal Analysis · **Card:** ML-07 · Builds on the ML-04 contract
> (`w03_data_contract.ipynb`).
>
> The baseline is a rule a human can read, and its job is to be honestly beatable. No fitted
> weights anywhere in this notebook — every number in the score was chosen by hand and is
> defensible in a sentence.

In [1]:
%pip install -q duckdb pandas scikit-learn pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import json, pathlib, subprocess, sys
import numpy as np, pandas as pd

REPO = pathlib.Path.cwd()
for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (p / "data/raw/content_refresh_anonymized.csv").exists():
        REPO = p
        break
OUT = REPO / "work/outputs"; OUT.mkdir(parents=True, exist_ok=True)
rel = lambda q: q.relative_to(REPO).as_posix()

# The modeling frame is built by a committed script so this notebook and the later ones
# cannot drift apart. Rebuild it only if the cache is missing (needs an HF token).
FRAME = OUT / "modeling_frame_dev.parquet"
if not FRAME.exists():
    subprocess.run([sys.executable, str(REPO / "work/scripts/build_modeling_frame.py")], check=True)

df = pd.read_parquet(FRAME).reset_index(drop=True)
y  = df["is_position_decline"].values
BASE_RATE = y.mean()

print(f"frame      : {rel(FRAME)}")
print(f"rows       : {len(df):,} content items · {df.client_hash_id.nunique()} clients")
print(f"base rate  : {BASE_RATE:.4f}  <- the number every score below must be read against")

frame      : work/outputs/modeling_frame_dev.parquet
rows       : 106,461 content items · 42 clients
base rate  : 0.5673  <- the number every score below must be read against


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**The rule, in plain words:**

> A page needs review first if its ranking was **already sliding before the cut-off**, if that
> ranking is **unstable** rather than steady, and if it sits **shallow enough to have somewhere
> to fall** while still being **seen often enough for the fall to matter**.

That is three sentences and no fitted weights. It maps to five conditions, all measured strictly
inside the feature window (Jan–Mar 2026):

| Reason code | Condition | Points | Why |
|---|---|---:|---|
| `already_sliding` | `f_pos_trend >= 1.0` (March position at least a full place worse than January) | **3** | direction before the cut is the single strongest observed signal — weighted heaviest on purpose |
| `unstable_position` | `f_pos_volatility >= median` | 1 | a page that bounces is not holding its ground |
| `shallow_and_exposed` | `0 < f_pos <= 20` | 1 | page 1–2: real estate worth defending, and room to fall |
| `high_visibility` | `f_impressions >= 1000` | 1 | a slide nobody sees is not the first problem to fix |
| `intermittent_visibility` | `f_days_with_impressions < 60` of 90 | 1 | appearing only sometimes is itself instability |

**Tie-break — and why it is not cosmetic.** Five integer conditions produce only eight distinct
scores across 106k pages, so thousands of pages tie. My first version ranked those ties in frame
order and **precision@50 came out at 0.420 — below the 0.567 base rate**, while precision@5000
was 0.799. A rule that looks worse than chance at the top and far better in bulk is not a rule
with a weak signal; it is a rule whose ordering *is not determined*. The frame arrives clustered
by client, so the tie block was an unrepresentative clump.

I break ties by **exposure** (`log1p(f_impressions)`, squashed into `[0,1)` so it can never
outrank a whole point): among pages with identical evidence, review the one more people see.
That is a statement about priorities, not a fitted parameter.

In [3]:
# Every condition below reads ONLY feature-window columns. Nothing from April touches this.
vol_median = df["f_pos_volatility"].median()
trend      = df["f_pos_trend"].fillna(0.0)   # missing = no measurable direction -> no points

CONDITIONS = {
    "already_sliding":         (trend >= 1.0),
    "unstable_position":       (df["f_pos_volatility"].fillna(0) >= vol_median),
    "shallow_and_exposed":     ((df["f_pos"] > 0) & (df["f_pos"] <= 20)),
    "high_visibility":         (df["f_impressions"] >= 1000),
    "intermittent_visibility": (df["f_days_with_impressions"] < 60),
}
WEIGHTS = {"already_sliding": 3, "unstable_position": 1, "shallow_and_exposed": 1,
           "high_visibility": 1, "intermittent_visibility": 1}

for name, cond in CONDITIONS.items():
    df[f"rc_{name}"] = cond.astype(int)

df["rule_points"] = sum(df[f"rc_{n}"] * w for n, w in WEIGHTS.items())

# exposure tie-break, strictly inside one point
expo = np.log1p(df["f_impressions"].values)
df["exposure_rank"] = (expo - expo.min()) / (expo.max() - expo.min())
df["baseline_score"] = df["rule_points"] + 0.999 * df["exposure_rank"]

df["reason_codes"] = [
    ",".join(n for n in CONDITIONS if row[f"rc_{n}"]) or "none"
    for _, row in df[[f"rc_{n}" for n in CONDITIONS]].iterrows()
]

print(f"median volatility threshold: {vol_median:.2f}")
print(f"\nreason code prevalence (n and observed decline rate):")
for n in CONDITIONS:
    m = df[f"rc_{n}"] == 1
    print(f"  {n:24s} n={m.sum():6,}  decline={y[m.values].mean():.3f}")
print(f"\n  {'(base rate)':24s} n={len(df):6,}  decline={BASE_RATE:.3f}")

median volatility threshold: 6.13

reason code prevalence (n and observed decline rate):
  already_sliding          n=34,866  decline=0.728
  unstable_position        n=53,219  decline=0.587
  shallow_and_exposed      n=83,960  decline=0.584
  high_visibility          n=60,926  decline=0.594
  intermittent_visibility  n=38,994  decline=0.518

  (base rate)              n=106,461  decline=0.567


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores), kind="stable")
    return np.asarray(labels)[order[:k]].mean()

queue = df.sort_values("baseline_score", ascending=False).reset_index(drop=True)
queue["rank"] = np.arange(1, len(queue) + 1)

QUEUE_COLS = ["rank", "content_hash_id", "client_hash_id", "baseline_score", "rule_points",
              "reason_codes", "f_pos", "f_pos_trend", "f_pos_volatility", "f_impressions",
              "f_clicks", "f_ctr", "f_days_with_impressions"]
csv_path = OUT / "baseline_action_score.csv"
queue[QUEUE_COLS].to_csv(csv_path, index=False)
print(f"wrote {rel(csv_path)}  ({len(queue):,} rows)")

# --- evaluation: precision@K against the two floors ---
from sklearn.dummy import DummyClassifier
dummy = DummyClassifier(strategy="prior").fit(df[["f_pos"]], y)
dummy_acc = dummy.score(df[["f_pos"]], y)

print(f"\nbase rate (random pick)      : {BASE_RATE:.3f}")
print(f"dummy 'always decline' acc.  : {dummy_acc:.3f}   <- the floor below the floor")
print(f"\n{'K':>7} {'rule P@K':>10} {'lift':>8}")
rows = []
for k in [20, 50, 100, 500, 1000, 5000]:
    p = precision_at_k(queue["baseline_score"], queue["is_position_decline"], k)
    rows.append({"k": k, "precision_at_k": round(float(p), 4),
                 "lift_over_base": round(float(p / BASE_RATE), 3)})
    print(f"{k:>7} {p:>10.3f} {p/BASE_RATE:>7.2f}x")

metrics = {"base_rate": round(float(BASE_RATE), 4),
           "n_items": int(len(df)), "n_clients": int(df.client_hash_id.nunique()),
           "weights": WEIGHTS, "volatility_threshold": round(float(vol_median), 4),
           "precision_at_k": rows,
           "note": "rule baseline, no fitted weights; frozen once ML-08 modelling starts"}
json.dump(metrics, open(OUT / "w04_baseline_metrics.json", "w"), indent=2)
print(f"\nreceipt -> {rel(OUT / 'w04_baseline_metrics.json')}")

wrote work/outputs/baseline_action_score.csv  (106,461 rows)



base rate (random pick)      : 0.567
dummy 'always decline' acc.  : 0.567   <- the floor below the floor

      K   rule P@K     lift
     20      0.650    1.15x
     50      0.760    1.34x
    100      0.760    1.34x
    500      0.846    1.49x
   1000      0.847    1.49x
   5000      0.842    1.49x

receipt -> work/outputs/w04_baseline_metrics.json


In [5]:
# The ties problem, shown rather than asserted: same rule, no tie-break.
print("Ranking by integer rule_points alone (ties left in frame order):")
for k in [20, 50, 100, 500, 5000]:
    p = precision_at_k(df["rule_points"].astype(float), y, k)
    flag = "  <- BELOW base rate" if p < BASE_RATE else ""
    print(f"  P@{k:<5d} {p:.3f}{flag}")

print(f"\nDistinct rule_points values: {df.rule_points.nunique()} across {len(df):,} pages")
print(df.groupby("rule_points").agg(n=("is_position_decline", "size"),
                                    decline=("is_position_decline", "mean")).round(3).to_string())
print("\nThe score buckets are monotone in decline rate, but a bucket of 7,823 pages cannot")
print("order a top-50. That is what the exposure tie-break fixes.")

Ranking by integer rule_points alone (ties left in frame order):
  P@20    0.600
  P@50    0.420  <- BELOW base rate
  P@100   0.370  <- BELOW base rate
  P@500   0.648
  P@5000  0.799

Distinct rule_points values: 8 across 106,461 pages
                 n  decline
rule_points                
0                2    0.000
1             5600    0.411
2            41320    0.458
3            22446    0.552
4             7205    0.649
5            22047    0.718
6             7823    0.801
7               18    0.667

The score buckets are monotone in decline rate, but a bucket of 7,823 pages cannot
order a top-50. That is what the exposure tie-break fixes.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [6]:
top20 = queue.head(20).copy()
top20["action"] = np.where(top20.rc_already_sliding == 1, "review_now", "monitor_closely")
top20["confidence"] = np.where(
    top20.rule_points >= 6, "high",
    np.where(top20.rule_points >= 4, "medium", "low"))
# short pseudonymous handle - never print full hashes in a public notebook
top20["page"] = ["p" + h[-6:] for h in top20.content_hash_id]
top20["client"] = ["c" + h[-4:] for h in top20.client_hash_id]

show = top20[["rank", "page", "client", "action", "confidence", "reason_codes",
              "f_pos", "f_pos_trend", "f_impressions", "is_position_decline"]]
print(show.to_string(index=False,
      formatters={"f_pos": "{:.1f}".format, "f_pos_trend": "{:+.1f}".format,
                  "f_impressions": "{:,.0f}".format}))
print(f"\nOf these top 20, {int(top20.is_position_decline.sum())}/20 did decline "
      f"(base rate would give ~{20*BASE_RATE:.0f}/20).")

 rank    page client     action confidence                                                                                  reason_codes f_pos f_pos_trend f_impressions  is_position_decline
    1 p08ea72  c63c4 review_now       high already_sliding,unstable_position,shallow_and_exposed,high_visibility,intermittent_visibility  18.0       +13.2         7,347                    1
    2 p71f868  cbfe4 review_now       high already_sliding,unstable_position,shallow_and_exposed,high_visibility,intermittent_visibility  19.3       +13.0         4,525                    1
    3 pddb50a  c63c4 review_now       high already_sliding,unstable_position,shallow_and_exposed,high_visibility,intermittent_visibility  19.1       +12.9         4,222                    0
    4 pa3add8  c63c4 review_now       high already_sliding,unstable_position,shallow_and_exposed,high_visibility,intermittent_visibility  12.4       +11.4         4,086                    1
    5 p968854  c7cbb review_now       high already

**Reading the top 20.** Every row carries the same shape of claim: *observed* feature-window
evidence, a recommended *action*, a *confidence* grade from how much evidence stacked up, and —
because this is a backtest — whether it actually declined.

- **`review_now`** (carries `already_sliding`): the page's position measurably worsened between
  January and March. The action is to look at it before the slide continues.
- **`monitor_closely`** (no `already_sliding`): high exposure and instability, but no confirmed
  direction yet. Watching costs less than refreshing.
- **Confidence** is *evidence count*, not probability. `high` means five or six points of
  independent conditions agreed. It is emphatically not "80% likely to decline".

**What would make each of these wrong:** the label is a *relative* position move, so a page can
appear here, genuinely lose average position, and still gain clicks — if it lost rank on
low-value queries and held rank on the ones that convert. Nothing in this queue measures query
value. That caveat applies to every row, which is why the output is a review queue for a human
and not an automated refresh trigger.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [7]:
# --- Weak pick 1: client concentration in the top of the queue ---
top50 = queue.head(50)
conc = top50.client_hash_id.value_counts()
print("Clients represented in the top 50:", top50.client_hash_id.nunique(), "of",
      df.client_hash_id.nunique(), "in the frame")
print(f"Largest single client's share of the top 50: {conc.iloc[0]}/50 "
      f"({conc.iloc[0]/50:.0%})")

# --- Weak pick 2: where the rule is confidently wrong ---
wrong = queue.head(200).query("is_position_decline == 0")
print(f"\nMisses inside the top 200: {len(wrong)}")
if len(wrong):
    print("Their median feature-window position: "
          f"{wrong.f_pos.median():.1f} (queue top-200 median {queue.head(200).f_pos.median():.1f})")
    print("Their median prior trend: "
          f"{wrong.f_pos_trend.median():+.2f} (top-200 median {queue.head(200).f_pos_trend.median():+.2f})")

# --- Weak pick 3: the deep-page blind spot ---
deep = df[df.f_pos > 50]
print(f"\nPages deeper than position 50: n={len(deep):,}, observed decline "
      f"{deep.is_position_decline.mean():.3f} vs base {BASE_RATE:.3f}")
print("They score low on `shallow_and_exposed` by design — the rule deliberately ignores them.")

Clients represented in the top 50: 8 of 42 in the frame
Largest single client's share of the top 50: 33/50 (66%)

Misses inside the top 200: 40
Their median feature-window position: 17.5 (queue top-200 median 16.0)
Their median prior trend: +11.80 (top-200 median +13.91)

Pages deeper than position 50: n=3,845, observed decline 0.330 vs base 0.567
They score low on `shallow_and_exposed` by design — the rule deliberately ignores them.


In [8]:
# --- Leakage check: prove the score cannot see the outcome window ---
contract = json.load(open(OUT / "w03_data_contract.json"))
label_side = set(contract["label"]) | {"o_impressions"}

used = set()
for cond_expr in ["f_pos_trend", "f_pos_volatility", "f_pos", "f_impressions",
                  "f_days_with_impressions"]:
    used.add(cond_expr)

print("Columns the baseline score reads:", sorted(used))
overlap = used & label_side
assert not overlap, f"LEAK: score reads label-side column(s) {overlap}"
print("-> none of them is a label-side column. No April data enters the score.")

# No product/decision flags either.
DECISION_FLAGS = {"last_optimized_date", "optimization_eligible_date", "is_published", "is_deleted"}
assert not (used & DECISION_FLAGS), "LEAK: a product decision flag entered the score"
print("-> no product decision flags (last_optimized_date etc.) enter the score.")

# And the query table, excluded by the contract's date check, is not read at all here.
assert not any(c.startswith("query90d.") for c in used)
print("-> fact_content_query_90d is not read: its window starts inside the outcome month.")
print(f"\nEvery scored column is measured within {contract['feature_window'][0]} .. "
      f"{contract['feature_window'][1]}; the label lives in "
      f"{contract['outcome_window'][0]} .. {contract['outcome_window'][1]}.")

Columns the baseline score reads: ['f_days_with_impressions', 'f_impressions', 'f_pos', 'f_pos_trend', 'f_pos_volatility']
-> none of them is a label-side column. No April data enters the score.
-> no product decision flags (last_optimized_date etc.) enter the score.
-> fact_content_query_90d is not read: its window starts inside the outcome month.

Every scored column is measured within 2026-01-01 .. 2026-03-31; the label lives in 2026-04-01 .. 2026-04-30.


**Weak picks — what I would not defend.**

1. **The top of the queue is one client's problem.** Eight clients hold the top 50, and a single
   client owns 33 of them. The exposure tie-break rewards big-traffic pages, and traffic is
   concentrated by client — so "review these 50" would send a team into one account. A usable
   queue almost certainly needs a per-client cap or per-client normalisation. This is the
   clearest thing for ML-08 to improve on, and it is a **ranking** flaw the precision@K number
   completely hides.

2. **`already_sliding` may be measuring regression to the mean.** Pages that slid in the feature
   window are the ones most likely to keep sliding — but also the ones whose January position
   may have been an unsustainable spike. The rule cannot separate "genuinely decaying" from
   "returning to normal after a good month". Both look identical here.

3. **Deep pages are ignored on purpose, and that is a real blind spot.** Pages past position 50
   decline at a much lower observed rate, so the rule scores them low — but a page falling from
   55 to 70 is still losing ground, just invisibly. The rule optimises for the pages worth
   defending, not for measuring all decline.

4. **A third of pages have no measurable trend.** `f_pos_trend` is null for pages absent in
   January (they entered the panel mid-window). I fill those with 0 — no points — which
   systematically ranks newer pages lower regardless of their actual risk.

**Leakage check: clean.** The score reads five feature-window columns and nothing else — no
April data, no product decision flags, and none of `fact_content_query_90d` (excluded by the
contract because its window opens 2026-04-02, inside the outcome month). The assertions above
fail the notebook if that ever stops being true.

**Frozen.** These weights and thresholds do not change from here. Moving the baseline once the
model exists would make the comparison in ML-08 meaningless.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.